<a href="https://colab.research.google.com/github/imprasanth03/python_DE_practice/blob/master/rdd_assessment_2_pyspark_(11_30).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("PySparkPractice").master("local[*]").getOrCreate()

sc = spark.sparkContext

employee_data = [
    (101, "Arun", "IT", 45000),
    (102, "Divya", "HR", 38000),
    (103, "Kavin", "IT", 62000),
    (104, "Meena", "Finance", 55000),
    (105, "Ravi", "IT", 29000),
    (106, "Priya", "HR", 48000),
    (107, "Sanjay", "Finance", 72000),
    (108, "Nisha", "IT", 51000),
    (109, "Vijay", "HR", 32000),
    (110, "Anu", "Finance", 42000)
]


employee_rdd = sc.parallelize(employee_data)

filtered_rdd = employee_rdd.filter(lambda x: x[2] == "IT" and x[3] > 40000)

result_rdd = filtered_rdd.map(
    lambda x: "Employee: " + x[1] +
    " | Department: " + x[2] +
    " | Original Salary: " + str(x[3]) +
    " | Revised Salary: " + str(x[3] + (x[3] * 10 // 100))
)

print(result_rdd.collect())



['Employee: Arun | Department: IT | Original Salary: 45000 | Revised Salary: 49500', 'Employee: Kavin | Department: IT | Original Salary: 62000 | Revised Salary: 68200', 'Employee: Nisha | Department: IT | Original Salary: 51000 | Revised Salary: 56100']


['Employee: Arun | Department: IT | Original Salary: 45000 | Revised Salary: 49500', 'Employee: Kavin | Department: IT | Original Salary: 62000 | Revised Salary: 68200', 'Employee: Nisha | Department: IT | Original Salary: 51000 | Revised Salary: 56100']


In [12]:
from pyspark.sql.functions import when


student_data = [
    (1, "Arun", 85, 78, 92),
    (2, "Divya", 45, 55, 60),
    (3, "Kavin", 35, 40, 38),
    (4, "Meena", 90, 88, 95),
    (5, "Ravi", 65, 72, 58),
    (6, "Priya", 30, 85, 80),
    (7, "Sanjay", 55, 60, 65),
    (8, "Nisha", 95, 92, 89),
    (9, "Vijay", 40, 35, 70),
    (10, "Anu", 75, 80, 85)
]

columns = ["student_id", "name", "python", "sql", "pyspark"]

student_df = spark.createDataFrame(student_data, columns)

student_df = student_df.withColumn(
    "total_marks",
    col("python") + col("sql") + col("pyspark")
)

student_df = student_df.withColumn(
    "average_marks",
    col("total_marks") / 3
)

student_df = student_df.withColumn(
    "result",
    when(
        (col("python") >= 40) &
        (col("sql") >= 40) &
        (col("pyspark") >= 40),
        "PASS"
    ).otherwise("FAIL")
)

student_df = student_df.withColumn(
    "grade", when(col("average_marks") >= 90, "A")
    .when(col("average_marks") >= 75, "B")
    .when(col("average_marks") >= 60, "C")
    .otherwise("D")
)

student_df.select(
    "name",
    "total_marks",
    "average_marks",
    "result",
    "grade"
).show()

+------+-----------+------------------+------+-----+
|  name|total_marks|     average_marks|result|grade|
+------+-----------+------------------+------+-----+
|  Arun|        255|              85.0|  PASS|    B|
| Divya|        160|53.333333333333336|  PASS|    D|
| Kavin|        113|37.666666666666664|  FAIL|    D|
| Meena|        273|              91.0|  PASS|    A|
|  Ravi|        195|              65.0|  PASS|    C|
| Priya|        195|              65.0|  FAIL|    C|
|Sanjay|        180|              60.0|  PASS|    C|
| Nisha|        276|              92.0|  PASS|    A|
| Vijay|        145|48.333333333333336|  FAIL|    D|
|   Anu|        240|              80.0|  PASS|    B|
+------+-----------+------------------+------+-----+



In [13]:
student_df.filter(
    (col("result") == "PASS") &
    (col("average_marks") >= 75)
).show()

+----------+-----+------+---+-------+-----------+-------------+------+-----+
|student_id| name|python|sql|pyspark|total_marks|average_marks|result|grade|
+----------+-----+------+---+-------+-----------+-------------+------+-----+
|         1| Arun|    85| 78|     92|        255|         85.0|  PASS|    B|
|         4|Meena|    90| 88|     95|        273|         91.0|  PASS|    A|
|         8|Nisha|    95| 92|     89|        276|         92.0|  PASS|    A|
|        10|  Anu|    75| 80|     85|        240|         80.0|  PASS|    B|
+----------+-----+------+---+-------+-----------+-------------+------+-----+



In [14]:
student_df.filter(
    (col("result") == "PASS") &
    (col("average_marks") >= 75)
).select(
    "name",
    "total_marks",
    "average_marks",
    "result",
    "grade"
).show()

+-----+-----------+-------------+------+-----+
| name|total_marks|average_marks|result|grade|
+-----+-----------+-------------+------+-----+
| Arun|        255|         85.0|  PASS|    B|
|Meena|        273|         91.0|  PASS|    A|
|Nisha|        276|         92.0|  PASS|    A|
|  Anu|        240|         80.0|  PASS|    B|
+-----+-----------+-------------+------+-----+

